In [ ]:
# Out-of-time validation on next month dataset
# Новый датасет: base_month = 2026-02-01, target_month = 2026-03-01

oot_df = pd.read_parquet("df_2026-02-01.parquet")  # поменяй путь на свой файл

oot_df["contact_id"] = oot_df["contact_id"].astype("int64")

# DAC segment, если нужен для срезов
if "dac_segment_12m" in oot_df.columns:
    oot_df["segment"] = oot_df["dac_segment_12m"].astype(str)

# recency fill как в train
recency_cols = [
    "cheque_recency",
    "login_recency",
    "omni_qr_recency",
    "omni_features_recency",
    "perf_recency",
]

for col in recency_cols:
    if col in oot_df.columns:
        oot_df[col] = oot_df[col].fillna(999)

# Проверяем, что все финальные фичи есть в новом датасете
missing_features = sorted(set(final_best_features) - set(oot_df.columns))
assert len(missing_features) == 0, f"Нет фичей в OOT датасете: {missing_features}"

# Проверяем, что таргет есть для оценки качества
assert target in oot_df.columns, f"Нет target в OOT датасете: {target}"

# Скоринг уже обученной модели
oot_score_col = "oot_score"
oot_df[oot_score_col] = final_best_model.predict_proba(oot_df[final_best_features])[:, 1]

# Метрики на out-of-time
oot_threshold = final_threshold if "final_threshold" in globals() else 0.3
oot_metrics = calculate_binary_metrics(
    oot_df[target],
    oot_df[oot_score_col],
    oot_threshold,
)

print("OOT metrics:")
display(pd.DataFrame(oot_metrics, index=["oot"]).T)

print("OOT target rate:", oot_df[target].mean())
print("OOT PR-AUC:", sk_average_precision_score(oot_df[target], oot_df[oot_score_col]))
print("OOT ROC-AUC:", sk_roc_auc_score(oot_df[target], oot_df[oot_score_col]))

# Децили score
oot_deciles = decile_report(oot_df, oot_score_col, target)
display(oot_deciles)

# Метрики по DAC-сегментам
if "segment" in oot_df.columns:
    oot_segment_rows = []

    for segment_name, part in oot_df.groupby("segment"):
        if part[target].nunique() < 2:
            continue

        oot_segment_rows.append({
            "segment": segment_name,
            "rows": len(part),
            "target_rate": part[target].mean(),
            "pr_auc": sk_average_precision_score(part[target], part[oot_score_col]),
            "roc_auc": sk_roc_auc_score(part[target], part[oot_score_col]),
        })

    oot_segment_metrics = pd.DataFrame(oot_segment_rows).sort_values("pr_auc", ascending=False)
    display(oot_segment_metrics)

In [ ]:
from catboost import CatBoostClassifier
from pathlib import Path

final_best_model = CatBoostClassifier()
final_best_model.load_model("final_best_model_churn_from_dac.cbm")

final_best_features = Path("final_best_features.txt").read_text(encoding="utf-8").splitlines()

final_threshold = 0.3